#### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

#### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


### Message based summarization
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("messages", 10), # summarize when message length is 10
            keep=("messages", 5) # keep last 5 messages in memory
        )
    ]
)


In [7]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [8]:
# Alternative test data

questions = [
    "what is 2+2?",
    "what is 3+3?",
    "what is 10*5?",    
    "what is 100/2?",
    "what is 50-25?",
    "what is 5^2?",
    "what is 10%3?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config=config)

    print(f"message: {response}")
    print(f"Message length: {len(response['messages'])}")

message: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='f05d7f5c-bee5-4af7-bd63-24fa0e88c0a8'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4.', 'extras': {'signature': 'El4KXAERTTIPmXyS7HnGryn2pJZE1iPeHJWUXVgA0IFOLKfRiQn/33MMGiM9ujdBqwM6uHteQCWmeYISIJRg8nLN0OjUoKPFTk8nj+LDmWx8N7vRhpx6USUqZhmh9zXf'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a080ac-1c51-7251-9977-052aeba3cce2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})]}
Message length: 2
message: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='f05d7f5c-bee5-4af7-bd63-24fa0e88c0a8'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4.', 'extras': {'signature': 'El4KXAERTTIPm

Summarization Based On Token Size

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
        1. Grand Hotel - 5 stars, $200/night, spa, pool, gym
        2. City Inn - 3 stars, $100/night, business center, free breakfast
        3. Budget Stay - 2 stars, $50/night, free parking, no breakfast
    """


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("tokens", 550), # summarize when token length is 550
            keep=("tokens", 200) # keep last 200 tokens in memory
        )
    ],
)


config={"configurable":{"thread_id":"test-2"}}


# token counter (approximate)
def count_tokens(messages):
    total_char = sum(len(str(m.content)) for m in messages)
    return total_char // 4  # approximate 4 chars per token



In [10]:
## Run test

cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"search hotels in {city}")]}, config=config)

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

New York: ~179 tokens, 4 messages
[HumanMessage(content='search hotels in New York', additional_kwargs={}, response_metadata={}, id='1f1cb64d-f0eb-452e-9923-c6420f90a8e3'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_64008': 'El4KXAERTTIPzR6fefChp0VpbWiZEUls+IZZtCNW/gaobmiPcjIb24vJdInAVAOTggNnbf/CAzwKGud4gtX6gUfq5IJH0dRPbX0TA5IvvmH0gzWudglLK4Y5UeVtf0FV'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a080b3-3a3f-7cf0-b58b-32aaefb18cd5-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'New York'}, 'id': 'call_64008', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 17, 'total_tokens': 70, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in New York:\n        1. Grand Ho

Summarization based on Fraction

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-3"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test with multiple cities
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

# Run test
for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~102 tokens (0.0797%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='801cb970-1dd9-46ec-8915-c555492968be'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_217843': 'El4KXAERTTIP9CCDbJhHJ0ymHqOUC130Jll3pLpU+gHZ6yxbW7Fb51ktHcVyYN79flhOvVxqE7Jl6qjYH37+D8xYZZwKSxzvkTabmXR6SYpjxnT4zEUGmdNEw9ZflIDw'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a080b5-ec7d-75d0-8a7d-aaba1567f22e-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_217843', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 16, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris: Grand Hotel $350, City Inn $180, B

### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [14]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )   
    ]
)

In [15]:
config = {"configurable": {"thread_id": "test-approve"}}

# Step 1. Request 
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]}, config=config
)

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e8abc88c-0c42-4ed6-b1b3-2b6939794f42'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "subject": "Hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_632370': 'El4KXAERTTIPqcIjY1fuofcsQma4evSIYv/SkV0UhGoWOznXZ4mJOwwlNTlI4RIw++8Wqt4TKESKVl+9V9yotVzSnkzOK4K0SpaeuAajnP10E82Cas9rc52LIS3dv/Xm'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a080be-becd-74e0-b6ef-42e2d37b1226-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'subject': 'Hello', 'recipient': 'john@test.com'}, 'id': 'call_632370', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

In [16]:
from langgraph.types import Command

# Step 2. Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "Email sent successfully to john@test.com with subject 'Hello'.", 'extras': {'signature': 'El4KXAERTTIPQkzaLQh+u86j/qtD+00ABBNmeuzHcaZK1/bq0DpU0vmXK/GEkfkfUE8/mrVUDqMP2V1DAhpBH0IBzQfE40r7Oq+kqC9dq9LPspxexKmcIjJctvSmwOOE'}}]


In [17]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e8abc88c-0c42-4ed6-b1b3-2b6939794f42'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "subject": "Hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_632370': 'El4KXAERTTIPqcIjY1fuofcsQma4evSIYv/SkV0UhGoWOznXZ4mJOwwlNTlI4RIw++8Wqt4TKESKVl+9V9yotVzSnkzOK4K0SpaeuAajnP10E82Cas9rc52LIS3dv/Xm'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a080be-becd-74e0-b6ef-42e2d37b1226-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'subject': 'Hello', 'recipient': 'john@test.com'}, 'id': 'call_632370', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

#### Reject

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [19]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [ ]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    # user rejects the action
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

    # Tool Message -> User rejected the tool call for `send_email_tool` with id call_222669

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'The user rejected the request to send the email, so the email was not sent.', 'extras': {'signature': 'El4KXAERTTIPIShaXr0Ulb58+ZlxgYR+o0OcDvHg7DQ0duBCoa5D6vdrqiVG6q5fdgvJkl8+29UStv+IGU0G2JuCv2kGd5wMDv8TpzYYGdNY6NBcacMifb3trVycG524'}}]


In [21]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7c43653c-c22c-4a75-ada8-74970861fd5a'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_222669': 'El4KXAERTTIP6RIxkXcWSNAfjxGSfQJIqQj4SXBW69nUVq+CefGZC7SbynnCBqId7NwUPIYi+f9AX/3aoSHC9AM2pksigreerOKAMe+sojwBO2Jqw6xEJfA3Le90aIyo'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a080c1-fb1f-7f02-97f3-03c9bcace656-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'call_222669', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

#### Editing

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [23]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='49fd3a23-2189-4da4-acd4-818571a58c18'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_207238': 'El4KXAERTTIPGBAryPYnXZKtmEQ09J5qok/CMQwmHod0DpprjJBnBV/udtM3cFCNWvH77HYpZmAt6XBc2EPWdnmNSyZx/kGHT8nUO1ODem3dNX88arrRCzRGTWIiAW8I'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a080c3-a080-7013-9182-abc075d83b4e-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_207238', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

In [25]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: [{'type': 'text', 'text': "Email sent to correct@email.com with subject 'Corrected Subject'", 'extras': {'signature': 'El4KXAERTTIPmX/w3Jv3wPD2Ji5n/Ji5FE0htRIxi0HE7KxkD0BUg1bLSFbY1FNaCqVE4ZvTkTRRVYQKeKMDVLUZsnL3M2v8raCFHOU8PZP6DXmC3i+8iyUa/PQzv8gP'}}]


In [27]:
result

# Tool Message: Email sent to correct@email.com with subject 'Corrected Subject'

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='49fd3a23-2189-4da4-acd4-818571a58c18'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_207238': 'El4KXAERTTIPGBAryPYnXZKtmEQ09J5qok/CMQwmHod0DpprjJBnBV/udtM3cFCNWvH77HYpZmAt6XBc2EPWdnmNSyZx/kGHT8nUO1ODem3dNX88arrRCzRGTWIiAW8I'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a080c3-a080-7013-9182-abc075d83b4e-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Corrected Subject', 'body': 'This was edited by human before sending'}, 'id': 'call_207238'}], invalid_tool_calls=[